In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("../data/raw/data.csv")

In [3]:
df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"])

df["transaction_hour"] = df["TransactionStartTime"].dt.hour
df["transaction_day"] = df["TransactionStartTime"].dt.day
df["transaction_month"] = df["TransactionStartTime"].dt.month
df["transaction_weekday"] = df["TransactionStartTime"].dt.weekday

In [5]:
encoded = pd.get_dummies(
    df[["ProductCategory", "ChannelId"]],
    drop_first=True
)

# Add CustomerId back before grouping
encoded["CustomerId"] = df["CustomerId"].values

encoded = encoded.groupby("CustomerId").sum().reset_index()

In [7]:
customer_features = df.groupby("CustomerId").agg(
    transaction_count=("TransactionId", "size"),
    total_amount=("Amount", "sum"),
    avg_amount=("Amount", "mean"),
    total_value=("Value", "sum"),
    fraud_rate=("FraudResult", "mean")
).reset_index()

final_df = customer_features.merge(
    encoded,
    on="CustomerId",
    how="left"
)

In [8]:
scaler = StandardScaler()

numeric_cols = final_df.select_dtypes(include=np.number).columns

final_df[numeric_cols] = scaler.fit_transform(
    final_df[numeric_cols]
)

In [9]:
final_df.to_csv(
    "../data/processed/engineered_features.csv",
    index=False
)